In [4]:
import glob
import yaml
import spacy
import pandas as pd
import spacy
from sentence_transformers import SentenceTransformer
import torch
from tqdm import tqdm
import weaviate
from collections import defaultdict
import json
import re
from bs4 import BeautifulSoup
from huggingface_hub import hf_hub_download, list_repo_files

/Applications/anaconda3/envs/fasthtml/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [5]:
files = glob.glob("../data/06_ner_cleaned/*.html")
len(files)

979

In [6]:
x=0
for file in files:
    with open(file, "r") as f:
        html = f.read()
    html = html.replace("http://collections.ushmm.org Contact reference@ushmm.org", " ")
    html = html.replace("collection Interview with ", " ")
    collections = html.count("collection Interview with")
    if not collections:
        print(file, collections)
        x=x+1
print(x)


../data/06_ner_cleaned/RG-50.030.0469_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.549.03.0004_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.030.0117_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.233.0103_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.030.0703_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.030.0093_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.030.0146_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.030.0752_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.030.0687_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.549.02.0043_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.233.0074_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.106.0147_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.030.0674_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.030.0060_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.106.0116_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.549.02.0012_trs_en_cleaned.html 0
../data/06_ner_cleaned/RG-50.03

In [16]:
import re
from collections import defaultdict

def find_repeated_sequences(files, sequence_length=7, min_occurrences=5):
    sequence_counter = defaultdict(int)
    
    for file in tqdm(files, desc="Processing files"):
        with open(file, "r") as f:
            html = f.read()
        
        # Remove unwanted text and clean up the content
        html = html.replace("http://collections.ushmm.org Contact reference@ushmm.org", " ")
        html = html.replace("collection Interview with ", " ")
        
        # Extract text content from HTML
        soup = BeautifulSoup(html, 'html.parser')
        text = soup.get_text()
        
        # Tokenize the text into words
        words = re.findall(r'\b\w+\b', text.lower())
        
        # Find sequences of 5 words and count their occurrences
        file_sequences = defaultdict(int)
        for i in range(len(words) - sequence_length + 1):
            sequence = ' '.join(words[i:i+sequence_length])
            file_sequences[sequence] += 1
        
        # Add sequences that appear more than min_occurrences times to the global counter
        for sequence, count in file_sequences.items():
            if count > min_occurrences:
                sequence_counter[sequence] += count
    
    return sequence_counter

# Usage
sequence_counter = find_repeated_sequences(files)

# Print the results
for sequence, count in sorted(sequence_counter.items(), key=lambda x: x[1], reverse=True):
    print(f"{sequence}: {count}")

Processing files: 100%|██████████| 979/979 [00:48<00:00, 20.01it/s]

collections ushmm org contact reference ushmm org: 6358
ushmm org contact reference ushmm org for: 6358
org contact reference ushmm org for further: 6358
contact reference ushmm org for further information: 6358
reference ushmm org for further information about: 6358
ushmm org for further information about this: 6358
http collections ushmm org contact reference ushmm: 3567
https collections ushmm org contact reference ushmm: 2791
it has not been checked for spelling: 2441
not been checked for spelling nor verified: 1846
been checked for spelling nor verified for: 1846
checked for spelling nor verified for accuracy: 1846
for spelling nor verified for accuracy this: 1846
spelling nor verified for accuracy this document: 1846
nor verified for accuracy this document should: 1846
verified for accuracy this document should not: 1846
for accuracy this document should not be: 1846
accuracy this document should not be quoted: 1846
this document should not be quoted or: 1846
document should not 

In [17]:
# Print the results
for sequence, count in sorted(sequence_counter.items(), key=lambda x: x[1], reverse=True):
    if count > 100:
        print(f"{sequence}: {count}")

Processing files: 100%|██████████| 979/979 [00:49<00:00, 19.95it/s]

collections ushmm org contact reference ushmm org: 6358
ushmm org contact reference ushmm org for: 6358
org contact reference ushmm org for further: 6358
contact reference ushmm org for further information: 6358
reference ushmm org for further information about: 6358
ushmm org for further information about this: 6358
http collections ushmm org contact reference ushmm: 3567
https collections ushmm org contact reference ushmm: 2791
it has not been checked for spelling: 2441
not been checked for spelling nor verified: 1846
been checked for spelling nor verified for: 1846
checked for spelling nor verified for accuracy: 1846
for spelling nor verified for accuracy this: 1846
spelling nor verified for accuracy this document: 1846
nor verified for accuracy this document should: 1846
verified for accuracy this document should not: 1846
for accuracy this document should not be: 1846
accuracy this document should not be quoted: 1846
this document should not be quoted or: 1846
document should not 